# M0 GPU checks — prefix-fork on Qwen3.5-0.8B-Base

Runs the fork equivalence tests with the `flash-linear-attention` CUDA kernel and benchmarks fork vs. one full forward per branch.

**Runtime → Change runtime type → GPU** (T4 is fine), then **Runtime → Run all**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
%cd /content
!rm -rf qwen-rlcd && git clone -q -b main https://github.com/shamazharikh/qwen-rlcd
%cd /content/qwen-rlcd
!git log --oneline -1
!pip install -q -e '.[dev]'
!python -c "import transformers, system_one; print('transformers', transformers.__version__)"

## Install CUDA kernels
Only `flash-linear-attention` (Triton, no compile step) for the Gated DeltaNet kernel. `causal-conv1d` is skipped: compiling it on a Colab T4 VM ran out of memory and killed the runtime twice, and the kernel-4 short conv is cheap on the torch path.

In [ ]:
!pip install -q flash-linear-attention
!python -c "import importlib.util as u, torch, transformers, fla; print('torch', torch.__version__, '| transformers', transformers.__version__, '| fla', fla.__version__, '| causal_conv1d installed:', u.find_spec('causal_conv1d') is not None)"

## Fork equivalence tests on CUDA (tiny model + real weights; fp32, plus bf16 on Ampere+)

In [ ]:
!QWEN_RLCD_SLOW=1 python -m pytest -q -s 2>&1 | tee test.log | grep -vE 'Loading weights|Warning: You are sending'
!echo; grep 'falling back' test.log | sort -u | sed 's/^/FALLBACK: /'; grep -q 'chunk_gated_delta_rule. is falling back' test.log && echo 'WARNING: DeltaNet ran on the torch fallback, not fla' || echo 'OK: DeltaNet used the fla kernel'

## Benchmark: fork vs. sequential

In [ ]:
!python scripts/bench_fork.py 2>&1 | grep -vE 'Loading weights|Warning: You are sending|falling back'